# Homework 4: Anomaly Scores and Review Decisions

Suppose a laboratory can review 150 white-wine records. Use LOF and Isolation Forest to choose records, compare their selections with sensory ratings, and examine how one neighborhood choice changes the result. Begin with a small LOF calculation to connect the software scores to the method.

This individual assignment is worth **10 points** and is due **November 1 at 11:59 p.m. Central**. Complete the Isolation Forest modeling cell in Section 2 and the 10-neighbor LOF modeling cell in Section 4 and answer Questions 1a–4b briefly. Setup, data preparation, displays, and selection code are supplied.

Use the [09b notebook](https://github.com/olearydj/INSY7130/blob/main/lectures/09b-lof-and-isolation-forest/09b-lof-and-isolation-forest.ipynb), especially “Fit And Inspect LOF” and “Fit The Forest,” and the [10a notebook](https://github.com/olearydj/INSY7130/blob/main/lectures/10a-anomaly-decisions-and-validation/10a-anomaly-investigation.ipynb), especially “Score The Red-Wine Records,” “Set A Review Threshold,” and “Check Whether A Setting Changes The Decision.”

## 1. Calculate one LOF score

Use these invented values on a number line, with $k=2$: **0, 2, 5, 12, 40, 50, 70**. Each observation is excluded from its own neighbor set. Use the pseudocode below to calculate $\operatorname{LOF}_2(0)$. Calculate the required local reachability densities for 0 and its two nearest neighbors, 2 and 5.

Recall that the radius belongs to the **receiving neighbor** $o$:

$$\operatorname{reach}_2(p,o)=\max\{|p-o|,r_2(o)\}.$$

$$\operatorname{lrd}_2(p)=\frac{1}{\text{mean reachability distance to its two neighbors}}.$$

$$\operatorname{LOF}_2(p)=\frac{\text{mean local reachability density of its two neighbors}}{\operatorname{lrd}_2(p)}.$$

The process of calculating LOF can be described in three passes. Here, `radius[p]` denotes $r_k(p)$, the distance from $p$ to its kth-nearest neighbor; in this exercise, $k=2$. Similarly, `lrd[p]` and `lof[p]` denote $\operatorname{lrd}_k(p)$ and $\operatorname{LOF}_k(p)$ for the chosen $k$:

```text
INPUT:
    observations X
    neighborhood size k, where 1 ≤ k < number of observations
    distance function d

PASS 1: Find each observation's neighbors and radius
    for each observation p in X:
        neighbors[p] = k nearest observations to p, excluding p itself
        radius[p] = distance from p to its kth-nearest neighbor

PASS 2: Calculate local reachability density
    for each observation p in X:
        total_reach = 0

        for each neighbor o in neighbors[p]:
            reach = max(d(p, o), radius[o])
            total_reach = total_reach + reach

        mean_reach = total_reach / k
        lrd[p] = 1 / mean_reach

PASS 3: Calculate Local Outlier Factor
    for each observation p in X:
        total_neighbor_density = 0

        for each neighbor o in neighbors[p]:
            total_neighbor_density += lrd[o]

        mean_neighbor_density = total_neighbor_density / k
        lof[p] = mean_neighbor_density / lrd[p]

```

In this pseudocode:

```text
radius[o] depends on o's neighbors
reach(p, o) uses radius[o]
lrd[p] = 1 / mean reachability distance from p to its neighbors
lof[p] compares neighboring LRDs with p's own LRD
```

Show the arithmetic, using fractions or at least three decimal places. You may use a calculator or explicit Python arithmetic. Type your calculations below or attach a PDF or clear JPG/PNG photographs with your notebook. Label attachments 1a–1b and include your name.

**1a.** Use the pseudocode to calculate $\operatorname{LOF}_2(0)$. Show:

1. The two nearest neighbors and second-neighbor radius for each of 0, 2, and 5.
2. The reachability distances and LRD for each of these three observations.
3. The average LRD of 0's two nearest neighbors and the resulting LOF for 0.

> Replace this text with your answer to 1a.

**1b.** What does the LOF ratio say about 0's local support relative to its neighbors? Does it give a probability that the observation is faulty?

> Replace this text with your answer to 1b.

## 2. Select records for review

The prepared file contains 4,898 white-wine records from the [UCI Wine Quality dataset](https://archive.ics.uci.edu/dataset/186/wine+quality), credited to Cortez and colleagues (2009), DOI [10.24432/C56S3T](https://doi.org/10.24432/C56S3T), under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Each row records 11 chemical measurements and a sensory quality rating.

Residual sugar is measured in g/L, sulfur dioxide in mg/L, and alcohol in percent by volume. Density is recorded in g/cm³. The remaining features describe acidity, salts, and pH. `record_id` identifies source rows and is excluded from the detector inputs. `quality` is the median of at least three sensory assessments, on a 0–10 scale, and is excluded from the detector inputs.

Run the code cell below to load and standardize the dataset. Use the result for all subsequent calculations.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import IsolationForest  # noqa: F401 — used in modeling task
from sklearn.neighbors import LocalOutlierFactor  # noqa: F401 — used in modeling tasks
from sklearn.preprocessing import StandardScaler

DATA_URL = (
    "https://raw.githubusercontent.com/olearydj/INSY7130/"
    "main/homework/hw4/data/white-wine-records.csv"
)
candidates = [
    Path("data/white-wine-records.csv"),
    Path("homework/hw4/data/white-wine-records.csv"),
    Path("content/homework/hw4/data/white-wine-records.csv"),
]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is not None:
    wine = pd.read_csv(data_path).set_index("record_id")
else:
    with urlopen(DATA_URL, timeout=30) as response:
        wine = pd.read_csv(response).set_index("record_id")

chemistry = wine.drop(columns="quality")
assert wine.shape == (4898, 12) and not wine.isna().any().any()
assert chemistry.shape[1] == 11
scaler = StandardScaler()
X = scaler.fit_transform(chemistry)
BUDGET = 150
print(f"scikit-learn {sklearn.__version__}; {len(wine):,} records; {X.shape[1]} features")
preview_fields = ["residual sugar", "density", "alcohol", "quality"]
# Preview a few fields; X above still contains all 11 chemical features.
display(wine[preview_fields].head())

Run the following code cell to define the functions that select records and summarize outcome counts. `choose_reviews` finds the 150th-highest full-precision score and selects every record at or above it. Boundary ties are kept together, so a threshold can select more than the nominal budget. Use the actual selected counts displayed in the review summaries below in your answers. The threshold implements the laboratory’s review capacity.

In [ ]:
def choose_reviews(score, budget=BUDGET):
    cutoff = score.nlargest(budget).iloc[-1]
    return score.ge(cutoff)


def outcome_counts(chosen, low_rating):
    return {
        "Reviewed": int(chosen.sum()),
        "Low-rated found (TP)": int((chosen & low_rating).sum()),
        "Other reviewed (FP)": int((chosen & ~low_rating).sum()),
        "Low-rated missed (FN)": int((~chosen & low_rating).sum()),
        "Other unreviewed (TN)": int((~chosen & ~low_rating).sum()),
    }

Run the cell below to fit an LOF model with **20 neighbors**. The constructor sets the neighborhood size, `.fit(X)` calculates local support in this collection, and negating `negative_outlier_factor_` makes larger scores mean more unusual. This task scores the fitted collection, so we keep the default `novelty=False`. Use this fitted-object pattern and the lecture examples for the two modeling tasks that follow.

In [ ]:
lof_model = LocalOutlierFactor(n_neighbors=20)
lof_model.fit(X)
lof_scores = -lof_model.negative_outlier_factor_

In the code cell below, fit an Isolation Forest to the same `X`, using **300 trees**, **256 samples per tree**, and **random_state=7130**. Store it as `forest_model` and the larger-means-more-unusual scores as `isolation_scores`. Retrieve `score_samples(X)` and reverse its sign, as in the lecture.

In [ ]:
isolation_scores = None
# Fit the Isolation Forest and retrieve the correctly oriented score array.

Run the code cell below to display each detector’s review count, print the counts of shared and method-only selections, and plot the two anomaly scores. In the scatterplot, each point represents one record; the dashed lines mark the two review thresholds. Scores at or above a method’s threshold are selected. The LOF axis uses a logarithmic scale to keep the dense group and the largest scores visible. The threshold lines divide the plot into regions selected by both detectors, by one detector, or by neither.

In [ ]:
scores = flags = None
if lof_scores is None or isolation_scores is None:
    print("Complete the Isolation Forest cell above to compare review lists.")
else:
    scores = pd.DataFrame(
        {"LOF": lof_scores, "Isolation Forest": isolation_scores}, index=wine.index
    )
    flags = scores.apply(choose_reviews)
    display(pd.DataFrame({"Records selected": flags.sum()}))
    overlap = pd.Series({
        "LOF only": int((flags["LOF"] & ~flags["Isolation Forest"]).sum()),
        "Both methods": int(flags.all(axis=1).sum()),
        "Isolation Forest only": int((~flags["LOF"] & flags["Isolation Forest"]).sum()),
    })
    print("; ".join(f"{label}: {count}" for label, count in overlap.items()))
    thresholds = scores.apply(lambda column: column.nlargest(BUDGET).iloc[-1])
    fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
    ax.scatter(scores["LOF"], scores["Isolation Forest"], s=14, alpha=0.35, color="#245c91")
    ax.axvline(thresholds["LOF"], color="#333333", linestyle="--", label="LOF threshold")
    ax.axhline(thresholds["Isolation Forest"], color="#b65313", linestyle="--",
               label="Isolation Forest threshold")
    ax.set(xscale="log", xlabel="LOF score (log scale): more unusual →",
           ylabel="Isolation Forest score: more unusual →", title="Scores and review thresholds")
    ax.set_xticks([1, 1.5, 2, 3, 5, 7], labels=["1", "1.5", "2", "3", "5", "7"])
    ax.annotate("Record 2782",
                xy=(scores.loc[2782, "LOF"], scores.loc[2782, "Isolation Forest"]),
                xytext=(-18, -24), textcoords="offset points",
                ha="right", va="top", fontsize=10,
                arrowprops={"arrowstyle": "->", "color": "#333333"})
    ax.minorticks_off()
    ax.legend(loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()

**2.** In the scatterplot above, choose one region where the methods disagree. Identify it relative to the threshold lines, state which method selects those records, and report its record count from the method-only selection counts printed above the scatterplot. Using the methods’ scoring ideas, explain why the same records and review capacity need not produce the same selections. Two or three sentences are sufficient.

> Replace this text with your answer to 2.

## 3. Compare selections with sensory ratings

For this comparison, define **low-rated** as `quality <= 4`, following 10a.

Run the code cell below to display a table reporting how many selected records meet the low-rating criterion. **Precision** divides that number by the number reviewed; **recall** divides it by all low-rated records in the collection. The table also reports both percentages. To assess their usefulness, compare the observed yield with uniform random selection of the same number of records.

In [ ]:
low_rating = wine["quality"].le(4)
print("Low-rated records in the collection:", int(low_rating.sum()), "of", len(wine))
if flags is None:
    print("Complete Section 2 to display the outcome counts.")
else:
    counts = pd.DataFrame({
        method: outcome_counts(flags[method], low_rating) for method in flags
    }).T
    outcomes = counts[["Reviewed", "Low-rated found (TP)"]].copy()
    outcomes["Precision (%)"] = 100 * outcomes["Low-rated found (TP)"] / outcomes["Reviewed"]
    outcomes["Recall (%)"] = 100 * outcomes["Low-rated found (TP)"] / low_rating.sum()
    display(outcomes.round(1))

**3a.** Based on the number of low-rated records in the collection, how many low-rated records would you expect if 150 records were selected uniformly at random from the collection?

Based on the true-positive counts in the table above, do the detectors find more low-rated records than expected from random review? How does that comparison help you interpret their precision and recall? What relationship between chemical unusualness and rating does this suggest? Name one additional check before using a detector to find chemical faults in future wines. Three or four sentences after the calculations are sufficient.

> Replace this text with your answer to 3a.

Run the code cell below to display two tables for record 2782, labeled in the upper-right corner of the scatterplot above: its measurements and rating beside collection medians, and its anomaly rank under each detector. Compare this record’s sensory rating with its chemical measurements and anomaly ranks. The first table shows selected measurements in their original units alongside collection medians.

In [ ]:
profile_fields = ["residual sugar", "density", "volatile acidity", "quality"]
profile = pd.DataFrame({
    "Record 2782": wine.loc[2782, profile_fields],
    "Collection median": wine[profile_fields].median(),
})
display(profile)
if scores is not None:
    print("Rank 1 indicates the highest anomaly score:")
    display(scores.rank(ascending=False, method="min").loc[[2782]].astype(int))

**3b.** Does record 2782 meet the low-rating criterion? Does that establish whether its chemistry is typical? Use one measurement comparison from the first table above and the anomaly ranks from the second table to explain what you would investigate. Two or three sentences are sufficient.

> Replace this text with your answer to 3b.

## 4. Sensitivity to Neighborhood Size

Keep the same records, features, scaler, and capacity of 150. In the first code cell below, fit a second LOF model with **10 neighbors**, and save its larger-means-more-unusual score array as `lof10_scores`. Use a separate fitted model named `lof10_model` so the earlier scores remain available. Run the second code cell below to display the outcome counts for both neighborhood sizes and print how many records remain selected, enter, and leave the review list.

In [ ]:
lof10_scores = None
# Fit a second LOF model with 10 neighbors and retrieve its oriented scores.

In [ ]:
if flags is None or lof10_scores is None:
    print("Complete Sections 2 and 4 to compare the two neighborhoods.")
else:
    score10 = pd.Series(lof10_scores, index=wine.index)
    chosen10 = choose_reviews(score10)
    sensitivity = pd.DataFrame({
        "20 neighbors": outcome_counts(flags["LOF"], low_rating),
        "10 neighbors": outcome_counts(chosen10, low_rating),
    }).T
    display(sensitivity)
    common = int((flags["LOF"] & chosen10).sum())
    print("Records selected at both neighborhood sizes:", common)
    print("Records entering the list:", int((chosen10 & ~flags["LOF"]).sum()))
    print("Records leaving the list:", int((flags["LOF"] & ~chosen10).sum()))

**4a.** Using the counts printed above, how many records remain selected, enter, and leave when the neighborhood changes? Use the outcome table above to compare the number of low-rated wines found and calculate precision at the two settings. Does an unchanged review count establish a stable investigation list?

> Replace this text with your answer to 4a.

**4b.** Explain why changing `n_neighbors` requires a new LOF fit, whereas changing a threshold on existing scores does not. What part of the analysis changes in each case?

Suppose review capacity could change. If missing a low-rated wine were more costly than reviewing a wine rated above 4, would you raise or lower the threshold? How would your choice change if those costs were reversed?

> Replace this text with your answer to 4b.

## Check and submit

Submit your completed `.ipynb` with the Isolation Forest and 10-neighbor LOF modeling cells completed, answers to 1a, 1b, 2, 3a, 3b, 4a, and 4b, the tables, printed counts, and score scatterplot produced by the code cells, and your assistance disclosure. Include Part 1 in the notebook or as a separate PDF or clear JPG/PNG photographs in the same Canvas submission. If using attachments, write “See attached calculations” in the corresponding response blocks. Do not submit the CSV separately.

Restart the runtime and run all cells in order. Check that no “Complete...” reminders remain, that your calculations use the counts in the Section 3 and Section 4 output tables, and that your explanations agree with the tables, printed counts, and scatterplot produced by your notebook.

| Evaluation | Points |
| --- | ---: |
| Section 1: LOF calculation and meaning | 2.5 |
| Section 2: model application and review-list comparison | 2.0 |
| Section 3: random-review comparison, validation, and record interpretation | 3.0 |
| Section 4: sensitivity implementation and interpretation | 2.0 |
| Reproducibility and assistance disclosure | 0.5 |
| **Total** | **10.0** |

Short, accurate explanations are sufficient. You may use course materials, documentation, and LLM assistance for understanding and debugging. Your submitted calculations and explanations must be your own, and you must be able to explain or revise your code and reasoning. Identify material assistance and how you checked it, or write `None`.

**Assistance disclosure:**

> Replace this text with your answer to assistance disclosure.

**Optional, ungraded feedback:** Approximately how much active time did you spend, excluding breaks? Which part took the most effort, and was anything unclear?

> Optional feedback.